# DHPS MMGBSA — Reverse Molecular Design

5가지 방법(GA, BO, ScaffoldHop, Fragment, Retrieval)으로 역방향 분자 설계를 수행합니다.  
Seeds: 0-4, Targets: -30, -40 kcal/mol, N_generate: 50

In [ ]:
# --- Cell 1: Install packages ---
!pip install rdkit lightgbm torch scikit-learn scipy pandas transformers -q

In [ ]:
# --- Cell 2: Clone repo & setup ---
!git clone https://github.com/mschongchulshin/dhps-binding-prediction.git
import os
os.chdir('dhps-binding-prediction')
# dataset.csv를 repo 루트에 복사하거나 Google Drive에서 가져오세요
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset.csv .
!ls -la

In [ ]:
# --- Cell 2b: Install torch-geometric (Colab 전용) ---
!pip install torch-geometric -q

## Cell 3: Import, Data Load, Model Setup

data_utils monkey-patch + Ridge, LGB, GNN, BiLSTM 모델 학습.

In [ ]:
# --- Cell 3: Import & data load (monkey-patch load_data for CSV) ---
import os, sys, json, warnings, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")

# data_utils monkey-patch: load_data를 CSV 버전으로 교체
import data_utils

def load_data_csv(csv_path="dataset.csv"):
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog("rdApp.*")
    df = pd.read_csv(csv_path)
    df = df[["Smiles", "MMGBSA dG Bind"]].dropna()
    df["Smiles"] = df["Smiles"].astype(str).str.strip()
    df["MMGBSA dG Bind"] = pd.to_numeric(df["MMGBSA dG Bind"], errors="coerce")
    df = df.dropna()
    def canonicalize(s):
        mol = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(mol, canonical=True) if mol else None
    df["canonical_smiles"] = df["Smiles"].apply(canonicalize)
    df = df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)
    print(f"[load_data_csv] Loaded {len(df)} molecules")
    print(f"[load_data_csv] MMGBSA range: {df['MMGBSA dG Bind'].min():.2f} ~ {df['MMGBSA dG Bind'].max():.2f}")
    return df

data_utils.load_data = load_data_csv
from data_utils import split_data, get_rdkit_features

# RDKit imports
from rdkit import Chem, RDLogger, RDConfig
from rdkit.Chem import Descriptors, QED, AllChem, BRICS, RWMol, rdMolDescriptors
RDLogger.DisableLog("rdApp.*")

# SA Score
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
try:
    import sascorer
    SASCORER_AVAILABLE = True
except ImportError:
    print("[WARNING] sascorer not found; SA_score will be 0")
    SASCORER_AVAILABLE = False

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr, pearsonr
import lightgbm as lgb
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import AttentiveFP
from transformers import get_cosine_schedule_with_warmup
import model4_bilstm as m4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[5Method] Device: {DEVICE}")

RESULTS_DIR = "results"
TARGETS = [-30, -40]
SEEDS = [0, 1, 2, 3, 4]
N_GENERATE = 50
BASE_SEED = 42
METHODS = ["GA", "BO", "ScaffoldHop", "Fragment", "Retrieval"]

# Best params (하드코딩)
RIDGE_ALPHA = 3708.369222848137

LGB_PARAMS = {
    'n_estimators': 300,
    'max_depth': 8,
    'learning_rate': 0.21577860588866943,
    'num_leaves': 241,
    'subsample': 0.6914478144277556,
    'colsample_bytree': 0.48859784786222643,
    'reg_alpha': 0.005955621350112854,
    'reg_lambda': 0.24360262556508194,
    'min_child_samples': 50,
}

GNN_PARAMS = {
    'hidden_dim': 256,
    'num_layers': 2,
    'num_timesteps': 3,
    'dropout': 0.17238086369178526,
    'lr': 0.00246834608351206,
    'batch_size': 32,
}

BILSTM_PARAMS = {
    'embed_dim': 128,
    'hidden_dim': 512,
    'n_layers': 1,
    'dropout': 0.2366051379116888,
    'lr': 0.004749019991935565,
    'batch_size': 32,
}

os.makedirs(RESULTS_DIR, exist_ok=True)

def log(msg): print(msg, flush=True)

# ── GNN 정의 ──────────────────────────────────────────────────────────────────
NODE_DIM = 39
EDGE_DIM = 10

def mol_to_graph(smi, y=0.0):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumAtoms())
    atom_features = []
    for atom in mol.GetAtoms():
        common = [1,5,6,7,8,9,14,15,16,17,34,35,53]
        ohe = [int(atom.GetAtomicNum() == a) for a in common]
        f = ohe + [
            atom.GetDegree() / 6.0,
            atom.GetFormalCharge(),
            atom.GetNumImplicitHs() / 4.0,
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
        ]
        f = f[:NODE_DIM] + [0.0] * max(0, NODE_DIM - len(f))
        atom_features.append(f)
    if not atom_features:
        return None
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondTypeAsDouble()
        ring = int(bond.IsInRing())
        conj = int(bond.GetIsConjugated())
        stereo = int(bond.GetStereo())
        ef = [bt/3.0, ring, conj, stereo/5.0,
              int(bt==1), int(bt==1.5), int(bt==2), int(bt==3), int(ring), int(conj)]
        ef = ef[:EDGE_DIM] + [0.0] * max(0, EDGE_DIM - len(ef))
        for src, dst in [(i, j), (j, i)]:
            edge_index.append([src, dst])
            edge_attr.append(ef)
    if not edge_index:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y], dtype=torch.float))

class GNNRegressor(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_timesteps, dropout):
        super().__init__()
        self.gnn = AttentiveFP(
            in_channels=NODE_DIM, hidden_channels=hidden_dim,
            out_channels=hidden_dim, edge_dim=EDGE_DIM,
            num_layers=num_layers, num_timesteps=num_timesteps, dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, data):
        x = self.gnn(data.x, data.edge_index, data.edge_attr, data.batch)
        return self.head(x).squeeze(-1)

@torch.no_grad()
def _gnn_predict(model, smiles_list, device):
    model.eval()
    graphs = [mol_to_graph(s) for s in smiles_list]
    graphs = [g for g in graphs if g is not None]
    if not graphs:
        return np.array([])
    dl = PyGLoader(graphs, batch_size=128, shuffle=False)
    preds = []
    for batch in dl:
        batch = batch.to(device)
        preds.extend(model(batch).cpu().numpy().tolist())
    return np.array(preds)

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
log("[5Method] 데이터 로드...")
df = load_data_csv("dataset.csv")
df["mol_id"] = df.index
train_df, val_df, test_df = split_data(df, seed=BASE_SEED)
X_train = get_rdkit_features(train_df["canonical_smiles"].tolist())
y_train = train_df["MMGBSA dG Bind"].values
train_smiles_set = set(train_df["canonical_smiles"].tolist())
all_smiles = df["canonical_smiles"].tolist()

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

# ── Ridge 학습 ────────────────────────────────────────────────────────────────
log("[5Method] Ridge 학습...")
ridge = Ridge(alpha=RIDGE_ALPHA)
ridge.fit(X_train_sc, y_train)

# ── LightGBM 학습 ─────────────────────────────────────────────────────────────
log("[5Method] LightGBM 학습...")
lgb_model = lgb.LGBMRegressor(**LGB_PARAMS, random_state=BASE_SEED, verbosity=-1, n_jobs=-1)
lgb_model.fit(X_train, y_train)

# ── GNN 학습 ──────────────────────────────────────────────────────────────────
log("[5Method] GNN 학습...")
train_graphs_gnn = [g for g in
    [mol_to_graph(s, y) for s, y in zip(train_df["canonical_smiles"].tolist(), train_df["MMGBSA dG Bind"].tolist())]
    if g is not None]
val_graphs_gnn = [g for g in
    [mol_to_graph(s, y) for s, y in zip(val_df["canonical_smiles"].tolist(), val_df["MMGBSA dG Bind"].tolist())]
    if g is not None]

torch.manual_seed(BASE_SEED); np.random.seed(BASE_SEED)
gnn_model = GNNRegressor(
    GNN_PARAMS["hidden_dim"], GNN_PARAMS["num_layers"],
    GNN_PARAMS["num_timesteps"], GNN_PARAMS["dropout"]
).to(DEVICE)
gnn_optimizer = torch.optim.AdamW(gnn_model.parameters(), lr=GNN_PARAMS["lr"], weight_decay=1e-4)
_gnn_train_dl = PyGLoader(train_graphs_gnn, batch_size=GNN_PARAMS["batch_size"], shuffle=True)
_gnn_val_dl   = PyGLoader(val_graphs_gnn,   batch_size=128, shuffle=False)
_total = len(_gnn_train_dl) * 50
_sched = get_cosine_schedule_with_warmup(gnn_optimizer, int(_total * 0.05), _total)
_criterion = nn.MSELoss()
_best_rmse_gnn, _best_state_gnn, _pat = float("inf"), None, 0

for _ep in range(50):
    gnn_model.train()
    for _b in _gnn_train_dl:
        _b = _b.to(DEVICE)
        gnn_optimizer.zero_grad()
        _loss = _criterion(gnn_model(_b), _b.y)
        _loss.backward()
        nn.utils.clip_grad_norm_(gnn_model.parameters(), 1.0)
        gnn_optimizer.step()
        _sched.step()
    with torch.no_grad():
        gnn_model.eval()
        _vp, _vl = [], []
        for _b in _gnn_val_dl:
            _b = _b.to(DEVICE)
            _vp.extend(gnn_model(_b).cpu().numpy())
            _vl.extend(_b.y.cpu().numpy())
        _vrmse = float(np.sqrt(np.mean((np.array(_vp) - np.array(_vl)) ** 2)))
    if _vrmse < _best_rmse_gnn:
        _best_rmse_gnn = _vrmse
        _best_state_gnn = {k: v.cpu().clone() for k, v in gnn_model.state_dict().items()}
        _pat = 0
    else:
        _pat += 1
        if _pat >= 15:
            break

gnn_model.load_state_dict(_best_state_gnn)
log(f"  GNN best val RMSE={_best_rmse_gnn:.4f}")

# ── BiLSTM 학습 ───────────────────────────────────────────────────────────────
log("[5Method] BiLSTM 학습...")
aug_df = pd.read_pickle(f"{RESULTS_DIR}/augmented_full.pkl")
train_ids = set(train_df["mol_id"])
val_ids   = set(val_df["mol_id"])
train_aug_bl = aug_df[aug_df["mol_id"].isin(train_ids)].reset_index(drop=True)
val_orig_bl  = aug_df[aug_df["mol_id"].isin(val_ids) & (~aug_df["is_augmented"])].reset_index(drop=True)
bilstm_model, _ = m4.train_bilstm(
    train_aug_bl, val_orig_bl,
    embed_dim=BILSTM_PARAMS["embed_dim"],
    hidden_dim=BILSTM_PARAMS["hidden_dim"],
    n_layers=BILSTM_PARAMS["n_layers"],
    dropout=BILSTM_PARAMS["dropout"],
    lr=BILSTM_PARAMS["lr"],
    batch_size=BILSTM_PARAMS["batch_size"],
    epochs=10,
    device=DEVICE,
    seed=BASE_SEED,
)

# 전체 데이터 Ridge 예측 (Retrieval용)
X_all = get_rdkit_features(all_smiles)
X_all_sc = scaler.transform(X_all)
pred_all = ridge.predict(X_all_sc)

log("[5Method] 모든 모델 준비 완료")

## Cell 4: 5-Method Reverse Design (5-seed)

GA, BO, ScaffoldHop, Fragment, Retrieval 방법으로 각 target에 대해 분자를 생성합니다.

In [ ]:
# --- Cell 4a: evaluate_molecules 함수 정의 ---
def evaluate_molecules(smiles_list, target_dG):
    valid_smiles, valid_mols = [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            can = Chem.MolToSmiles(mol, canonical=True)
            valid_smiles.append(can)
            valid_mols.append(mol)
    if not valid_mols:
        return {"n": 0, "Ridge_hit3": 0.0, "LGB_hit3": 0.0,
                "GNN_hit3": 0.0, "BiLSTM_hit3": 0.0,
                "QED": 0.0, "SA_score": 0.0, "Lipinski": 0.0, "novelty": 0.0}

    X = get_rdkit_features(valid_smiles)
    X_sc = scaler.transform(X)
    pred_ridge  = ridge.predict(X_sc)
    pred_lgb    = lgb_model.predict(X)
    pred_gnn    = _gnn_predict(gnn_model, valid_smiles, DEVICE)
    pred_bilstm = m4.predict_forward_bilstm(bilstm_model, valid_smiles, DEVICE)

    qed_scores  = [QED.qed(m) for m in valid_mols]
    if SASCORER_AVAILABLE:
        sa_scores = [sascorer.calculateScore(m) for m in valid_mols]
    else:
        sa_scores = [0.0] * len(valid_mols)
    lipinski_ok = [
        int(Descriptors.MolWt(m) <= 500 and Descriptors.MolLogP(m) <= 5 and
            rdMolDescriptors.CalcNumHBD(m) <= 5 and rdMolDescriptors.CalcNumHBA(m) <= 10)
        for m in valid_mols
    ]
    novelty = sum(1 for s in valid_smiles if s not in train_smiles_set) / len(valid_smiles) * 100

    def hit_rate(preds):
        arr = np.array(preds, dtype=float)
        valid = arr[~np.isnan(arr)]
        if len(valid) == 0:
            return 0.0
        return round(sum(abs(p - target_dG) <= 3 for p in valid) / len(valid) * 100, 1)

    return {
        "n": len(valid_mols),
        "Ridge_hit3":  hit_rate(pred_ridge),
        "LGB_hit3":    hit_rate(pred_lgb),
        "GNN_hit3":    hit_rate(pred_gnn),
        "BiLSTM_hit3": hit_rate(pred_bilstm),
        "QED":      round(float(np.mean(qed_scores)), 3),
        "SA_score": round(float(np.mean(sa_scores)), 3),
        "Lipinski": round(float(np.mean(lipinski_ok)) * 100, 1),
        "novelty":  round(novelty, 1),
    }

In [ ]:
# --- Cell 4b: 5가지 역설계 방법 정의 ---

# ── GA ────────────────────────────────────────────────────────────────────────
def run_ga(target_dG, seed, pop_size=100, n_gen=150):
    random.seed(seed)
    train_mols = [Chem.MolFromSmiles(s) for s in train_df["canonical_smiles"]
                  if Chem.MolFromSmiles(s)]

    def fitness(smi):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return -999
        X = get_rdkit_features([smi])
        return -abs(ridge.predict(scaler.transform(X))[0] - target_dG)

    def mutate(mol):
        try:
            frags = BRICS.BRICSDecompose(mol)
            if len(frags) > 1:
                donor = random.choice(train_mols)
                donor_frags = BRICS.BRICSDecompose(donor)
                if donor_frags:
                    frag = random.choice(list(donor_frags))
                    new_mol = Chem.MolFromSmiles(
                        frag.replace('[*]','').replace('[1*]','').replace('[2*]','').replace('[3*]',''))
                    if new_mol:
                        return new_mol
        except:
            pass
        try:
            rwmol = RWMol(mol)
            atoms = [a for a in rwmol.GetAtoms() if a.GetAtomicNum() in [6,7,8]]
            if atoms:
                random.choice(atoms).SetAtomicNum(random.choice([6,7,8,9,16]))
            Chem.SanitizeMol(rwmol)
            return rwmol.GetMol()
        except:
            pass
        return mol

    pop = random.sample(train_mols, min(pop_size, len(train_mols)))
    pop_smi = [Chem.MolToSmiles(m) for m in pop]
    pop_fit = [fitness(s) for s in pop_smi]

    for _ in range(n_gen):
        children_smi = []
        for _ in range(pop_size // 2):
            i, j = random.sample(range(len(pop)), 2)
            parent = pop[i] if pop_fit[i] > pop_fit[j] else pop[j]
            children_smi.append(Chem.MolToSmiles(mutate(parent)))
        children_fit = [fitness(s) for s in children_smi]
        all_s = pop_smi + children_smi
        all_f = pop_fit + children_fit
        idx = np.argsort(all_f)[::-1][:pop_size]
        pop_smi = [all_s[i] for i in idx]
        pop_fit = [all_f[i] for i in idx]
        pop = [Chem.MolFromSmiles(s) or random.choice(train_mols) for s in pop_smi]

    top_idx = np.argsort(pop_fit)[::-1][:N_GENERATE]
    return [pop_smi[i] for i in top_idx]


# ── BO ────────────────────────────────────────────────────────────────────────
def run_bo(target_dG, seed):
    random.seed(seed)
    close_idx = np.argsort(np.abs(pred_all - target_dG))[:20]
    seed_mols = [Chem.MolFromSmiles(all_smiles[i]) for i in close_idx]
    seed_mols = [m for m in seed_mols if m]

    results = []
    for mol in seed_mols:
        for _ in range(50):
            try:
                rwmol = RWMol(mol)
                atoms = [a for a in rwmol.GetAtoms() if a.GetAtomicNum() in [6,7,8]]
                if atoms:
                    random.choice(atoms).SetAtomicNum(random.choice([6,7,8,9,16,17]))
                Chem.SanitizeMol(rwmol)
                new_smi = Chem.MolToSmiles(rwmol.GetMol())
                if Chem.MolFromSmiles(new_smi):
                    X = get_rdkit_features([new_smi])
                    pred = ridge.predict(scaler.transform(X))[0]
                    results.append((new_smi, abs(pred - target_dG)))
            except:
                continue
    for i in close_idx:
        smi = all_smiles[i]
        results.append((smi, abs(pred_all[i] - target_dG)))

    results.sort(key=lambda x: x[1])
    seen, unique = set(), []
    for smi, _ in results:
        if smi not in seen:
            seen.add(smi)
            unique.append(smi)
        if len(unique) >= N_GENERATE:
            break
    return unique


# ── Scaffold Hopping ─────────────────────────────────────────────────────────
def run_scaffoldhop(target_dG, seed):
    from rdkit.Chem.Scaffolds import MurckoScaffold
    random.seed(seed)
    np.random.seed(seed)

    close_idx = np.argsort(np.abs(pred_all - target_dG))[:30]
    seed_smiles = [all_smiles[i] for i in close_idx]

    substituents = set()
    for smi in random.sample(train_df["canonical_smiles"].tolist(), min(300, len(train_df))):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            try:
                frags = BRICS.BRICSDecompose(mol)
                for f in frags:
                    clean = f.replace('[*]','').replace('[1*]','').replace('[2*]','')\
                             .replace('[3*]','').replace('[4*]','')
                    m = Chem.MolFromSmiles(clean)
                    if m and m.GetNumAtoms() <= 8:
                        substituents.add(Chem.MolToSmiles(m, canonical=True))
            except:
                pass
    sub_list = list(substituents) if substituents else ["C", "N", "O", "F", "Cl"]

    results = []
    for smi in seed_smiles:
        mol = Chem.MolFromSmiles(smi)
        if not mol:
            continue
        try:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaf_smi = Chem.MolToSmiles(scaffold, canonical=True)
        except:
            scaf_smi = smi
        for _ in range(60):
            try:
                sub = random.choice(sub_list)
                combined = scaf_smi + "." + sub
                new_mol = Chem.MolFromSmiles(combined)
                if new_mol:
                    new_smi = Chem.MolToSmiles(new_mol, canonical=True)
                    X = get_rdkit_features([new_smi])
                    pred = ridge.predict(scaler.transform(X))[0]
                    results.append((new_smi, abs(pred - target_dG)))
            except:
                continue
        try:
            X = get_rdkit_features([smi])
            pred = ridge.predict(scaler.transform(X))[0]
            results.append((smi, abs(pred - target_dG)))
        except:
            pass

    results.sort(key=lambda x: x[1])
    seen, unique = set(), []
    for smi, _ in results:
        if smi not in seen:
            seen.add(smi)
            unique.append(smi)
        if len(unique) >= N_GENERATE:
            break
    return unique


# ── Fragment Assembly ─────────────────────────────────────────────────────────
def run_fragment(target_dG, seed):
    random.seed(seed)
    np.random.seed(seed)

    all_frags = set()
    for smi in train_df["canonical_smiles"].tolist():
        mol = Chem.MolFromSmiles(smi)
        if mol:
            try:
                frags = BRICS.BRICSDecompose(mol)
                all_frags.update(frags)
            except:
                pass
    frag_list = list(all_frags)
    if not frag_list:
        return []

    clean_frags = []
    for f in frag_list:
        s = f.replace('[*]','').replace('[1*]','').replace('[2*]','').replace('[3*]','').replace('[4*]','')
        mol = Chem.MolFromSmiles(s)
        if mol:
            clean_frags.append(Chem.MolToSmiles(mol, canonical=True))
    clean_frags = list(set(clean_frags))
    if not clean_frags:
        return []

    results = []
    for _ in range(10000):
        try:
            n = random.randint(1, 3)
            parts = random.sample(clean_frags, min(n, len(clean_frags)))
            combined = ".".join(parts)
            mol = Chem.MolFromSmiles(combined)
            if mol:
                smi = Chem.MolToSmiles(mol, canonical=True)
                X = get_rdkit_features([smi])
                pred = ridge.predict(scaler.transform(X))[0]
                results.append((smi, abs(pred - target_dG)))
        except:
            pass

    results.sort(key=lambda x: x[1])
    seen, unique = set(), []
    for smi, _ in results:
        if smi not in seen:
            seen.add(smi)
            unique.append(smi)
        if len(unique) >= N_GENERATE:
            break
    return unique


# ── Similarity Retrieval ──────────────────────────────────────────────────────
def run_retrieval(target_dG, seed):
    random.seed(seed)
    np.random.seed(seed)

    diffs = np.abs(pred_all - target_dG)
    idx = np.argsort(diffs)[:N_GENERATE * 3]

    candidates = []
    for i in idx:
        smi = all_smiles[i]
        mol = Chem.MolFromSmiles(smi)
        if mol:
            candidates.append(Chem.MolToSmiles(mol, canonical=True))
        if len(candidates) >= N_GENERATE:
            break

    random.shuffle(candidates)
    return candidates[:N_GENERATE]


log("[5Method] 함수 정의 완료")

In [ ]:
# --- Cell 4c: 5-seed × 5-method × 2-target 실행 ---
method_fns = {
    "GA": run_ga,
    "BO": run_bo,
    "ScaffoldHop": run_scaffoldhop,
    "Fragment": run_fragment,
    "Retrieval": run_retrieval,
}

seed_results = {str(t): {m: [] for m in METHODS} for t in TARGETS}

for seed in SEEDS:
    log(f"\n{'='*70}")
    log(f"[Seed {seed}]")
    log(f"{'='*70}")
    for target in TARGETS:
        log(f"  target={target}")
        for method in METHODS:
            t0 = time.time()
            try:
                smi_list = method_fns[method](target, seed)
                r = evaluate_molecules(smi_list, target)
            except Exception as e:
                log(f"    {method} ERROR: {e}")
                r = {"n": 0, "Ridge_hit3": 0.0, "LGB_hit3": 0.0,
                     "GNN_hit3": 0.0, "BiLSTM_hit3": 0.0,
                     "QED": 0.0, "SA_score": 0.0, "Lipinski": 0.0, "novelty": 0.0}
            seed_results[str(target)][method].append(r)
            log(f"    {method:<12} {time.time()-t0:.1f}s  "
                f"Ridge={r['Ridge_hit3']}%  LGB={r['LGB_hit3']}%  "
                f"GNN={r['GNN_hit3']}%  BiLSTM={r['BiLSTM_hit3']}%  "
                f"QED={r['QED']:.3f}  novelty={r['novelty']}%")


# ── 통계 집계 ─────────────────────────────────────────────────────────────────
METRICS = ["Ridge_hit3", "LGB_hit3", "GNN_hit3", "BiLSTM_hit3", "QED", "SA_score", "Lipinski", "novelty"]
summary = {}
for tkey in seed_results:
    summary[tkey] = {}
    for method in METHODS:
        runs = seed_results[tkey][method]
        m_stat = {k: float(np.mean([r[k] for r in runs])) for k in METRICS}
        s_stat = {k: float(np.std( [r[k] for r in runs])) for k in METRICS}
        summary[tkey][method] = {"mean": m_stat, "std": s_stat, "seeds": runs}

out_path = f"{RESULTS_DIR}/reverse_5method_5seed_result.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
log(f"\n저장: {out_path}")

## Cell 5: Results Summary

target별, method별 hit rate (LGB, GNN, BiLSTM) + QED + novelty 요약 표 출력.

In [ ]:
# --- Cell 5: 결과 요약 출력 ---
print(f"\n{'='*100}")
print(f"{'Target':>7} {'Method':<14} {'LGB_hit':>10} {'GNN_hit':>10} {'BiLSTM_hit':>12} {'QED':>8} {'Novelty':>10}")
print(f"{'-'*100}")

for t in TARGETS:
    for m in METHODS:
        s = summary[str(t)][m]
        print(f"{t:>7} {m:<14} "
              f"{s['mean']['LGB_hit3']:>5.1f}±{s['std']['LGB_hit3']:>4.1f}  "
              f"{s['mean']['GNN_hit3']:>5.1f}±{s['std']['GNN_hit3']:>4.1f}  "
              f"{s['mean']['BiLSTM_hit3']:>6.1f}±{s['std']['BiLSTM_hit3']:>4.1f}  "
              f"{s['mean']['QED']:>5.3f}±{s['std']['QED']:>5.3f}  "
              f"{s['mean']['novelty']:>5.1f}±{s['std']['novelty']:>4.1f}")
    print()

print(f"{'='*100}")
print("\n완료! 결과는 results/reverse_5method_5seed_result.json 에 저장됨.")

# QED 분포 요약
print("\n[QED 요약]")
for t in TARGETS:
    for m in METHODS:
        s = summary[str(t)][m]
        print(f"  target={t}, {m:<14}: QED={s['mean']['QED']:.3f}±{s['std']['QED']:.3f}  "
              f"SA={s['mean']['SA_score']:.2f}±{s['std']['SA_score']:.2f}  "
              f"Lipinski={s['mean']['Lipinski']:.1f}%")